<a href="https://colab.research.google.com/github/nov-cpu/8730-project/blob/API_Pulling/API_Pulling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-search-results pandas openpyxl

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=31d8ebf06a0381f4b493a1c1a1bc9132ff1876664f939d2f07a3c3ad92d2da28
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [ ]:
import pandas as pd
from serpapi import GoogleSearch
import time
import json

# 1. Load the Data
excel_path = '/content/alt_fuel_stations (Jul 26 2026).xlsx'

# Read the Locations and Stations sheets
df_locations = pd.read_excel(excel_path, sheet_name='Locations (Entity Table)')
df_stations = pd.read_excel(excel_path, sheet_name='Stations (Parent Table)')

# Clean up column names to avoid trailing whitespace issues
df_locations.columns = df_locations.columns.str.strip()
df_stations.columns = df_stations.columns.str.strip()

# Merge the tables to associate the Station Name with its Address and Coordinates
df_merged = pd.merge(
    df_stations,
    df_locations,
    left_on='Location_ID (FK)',
    right_on='Location_ID (PK)'
)

# 2. Setup SerpAPI Data Collection
API_KEY = 'YOUR_API_KEY_HERE'
results_list = []

# Note: We are using .head(5) to test the first 5 records and avoid burning API credits.
# Remove .head(5) to run this across the entire dataset once you verify it works!
for index, row in df_merged.head(5).iterrows():
    station_name = row['Station_name']
    street = row['street_Address']
    city = row['City']
    state = row['State']
    lat = row['Latitude']
    lon = row['Longitude']

    # Construct a robust search query (e.g., "Ramada 1319 2nd St W Brooks AB")
    search_query = f"{station_name} {street} {city} {state}"

    params = {
      "engine": "google_maps",
      "q": search_query,
      "ll": f"@{lat},{lon},15z", # Centers the map search around the exact lat/lon
      "type": "search",
      "api_key": API_KEY
    }

    try:
        search = GoogleSearch(params)
        results = search.get_dict()

        place = None

        # Check if an EXACT match exists (place_results)
        if "place_results" in results:
            place = results["place_results"]

        # Fallback to checking if multiple local results exist
        elif "local_results" in results and len(results["local_results"]) > 0:
            place = results["local_results"][0]

        if place:
            # Extract the ratings, review count, and place information
            extracted_data = {
                "Station_ID": row.get('Station_ID  (PK)'), # Using .get() is safer
                "Station_name": station_name,
                "Google_Place_ID": place.get("place_id"),
                "Google_Title": place.get("title"),
                "Rating": place.get("rating"),
                "Reviews_Count": place.get("reviews"),
                "Type": place.get("type"),
                "Address": place.get("address"),
                "Operating_Hours": place.get("operating_hours", {}).get("open_now")
            }
            results_list.append(extracted_data)
            print(f"Successfully collected data for: {station_name}")
        else:
            print(f"No results found for: {search_query}")

    except Exception as e:
        print(f"Error fetching data for {search_query}: {e}")

    # Respect API rate limits by pausing briefly between requests
    time.sleep(1)

# 3. Save the Extracted Data
df_results = pd.DataFrame(results_list)

# Export to JSON: Ideal for loading into MongoDB to handle unstructured/semi-structured data
df_results.to_json('google_maps_station_ratings.json', orient='records', indent=4)

# Export to CSV: Ideal for cleaning and loading into MySQL
df_results.to_csv('google_maps_station_ratings.csv', index=False)

print("\nData collection complete! JSON and CSV files have been saved.")

Successfully collected data for: Ramada
Successfully collected data for: Davis Chevrolet
Successfully collected data for: Gasonic Instruments
Successfully collected data for: International Motor Cars
Successfully collected data for: Residence Inn

Data collection complete! JSON and CSV files have been saved.
